In [4]:
!pip install nltk
# pip install streamlit

  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nltk]1/2 [nltk]


In [ ]:
import os
import re
import pickle
import warnings
import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"



In [7]:
df_raw = pd.read_csv("Movie Reviews DataSet.csv")
df_raw.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
print("Dataset shape:", df_raw.shape)
print("Unique sentiments:", df_raw["sentiment"].unique())
print("\nNull values:\n", df_raw.isnull().sum())

Dataset shape: (5000, 2)
Unique sentiments: ['positive' 'negative']

Null values:
 review       0
sentiment    0
dtype: int64


In [ ]:
df = df_raw.copy()

# remove rows with missing values
df.dropna(inplace=True)

# remove duplicates by review text
df = df.drop_duplicates(subset=["review"]).reset_index(drop=True)

clean_csv_path = "Movie_Reviews_Cleaned.csv"
df.to_csv(clean_csv_path, index=False)
print(f"Cleaned dataset saved: {clean_csv_path}")
df.head()


✅ Cleaned dataset saved: Movie_Reviews_Cleaned.csv


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
# NLTK downloads + preprocessing setup
for pkg in ["punkt", "stopwords", "wordnet"]:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

# some environments fail on this; keep it safe
try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

STOP_SET = set(stopwords.words("english"))
LEM = WordNetLemmatizer()

def normalize_text(txt: str) -> str:
    """
    - strips html tags
    - removes punctuation/special symbols
    - lowercases
    - tokenizes
    - removes stopwords
    - lemmatizes
    """
    if txt is None:
        return ""
    txt = str(txt)

    txt = re.sub(r"<[^>]*>", " ", txt)             # remove HTML
    txt = re.sub(r"[^a-zA-Z0-9\s]", " ", txt)      # keep alnum + space
    txt = re.sub(r"\s+", " ", txt).strip().lower() # normalize spaces + lowercase

    toks = word_tokenize(txt)
    toks = [LEM.lemmatize(t) for t in toks if t not in STOP_SET]
    return " ".join(toks)


In [ ]:
# Apply preprocessing
df["cleaned_review"] = df["review"].astype(str).apply(normalize_text)

df = df[df["cleaned_review"].str.len() > 0].reset_index(drop=True)

df[["review", "cleaned_review", "sentiment"]].head()


,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode h...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...,positive


In [ ]:
# TF-IDF features (1-3 grams)
tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=5000
)
X_tfidf = tfidf_vec.fit_transform(df["cleaned_review"])
print(" TF-IDF matrix:", X_tfidf.shape)


✅ TF-IDF matrix: (4997, 5000)


In [ ]:
# LSTM Tokenization + Padding

VOCAB_LIMIT = 5000  
SEQ_LEN = 150       

print("🔄 Building tokenizer & sequences...")
tok = Tokenizer(num_words=VOCAB_LIMIT, oov_token="<OOV>")
tok.fit_on_texts(df["cleaned_review"])

seqs = tok.texts_to_sequences(df["cleaned_review"])
X_seq = pad_sequences(seqs, maxlen=SEQ_LEN, padding="post", truncating="post")

print("Padded sequences:", X_seq.shape)


🔄 Building tokenizer & sequences...
✅ Padded sequences: (4997, 150)


In [ ]:
# Encode labels

enc = LabelEncoder()
y = enc.fit_transform(df["sentiment"])

print("Label mapping:")
for idx, cls_name in enumerate(enc.classes_):
    print(f"  {cls_name} -> {idx}")
print("y shape:", y.shape)


Label mapping:
  negative -> 0
  positive -> 1
✅ y shape: (4997,)


In [ ]:
# Train/Test split 
Xtr_tfidf, Xte_tfidf, y_tr, y_te = train_test_split(
    X_tfidf, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Xtr_seq, Xte_seq, _, _ = train_test_split(
    X_seq, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train size:", Xtr_tfidf.shape[0], "| Test size:", Xte_tfidf.shape[0])


Train size: 3997 | Test size: 1000


In [ ]:

# Train Logistic Regression (TF-IDF)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

print("Training Logistic Regression...")
lr_clf = LogisticRegression(max_iter=1000, random_state=42)
lr_clf.fit(Xtr_tfidf, y_tr)

pred_lr = lr_clf.predict(Xte_tfidf)

acc_lr = accuracy_score(y_te, pred_lr)
pre_lr = precision_score(y_te, pred_lr, average="weighted", zero_division=0)
rec_lr = recall_score(y_te, pred_lr, average="weighted", zero_division=0)
f1_lr  = f1_score(y_te, pred_lr, average="weighted", zero_division=0)

print("\n" + "-"*55)
print("LOGISTIC REGRESSION (TF-IDF) — RESULTS")
print("-"*55)
print(f"Accuracy : {acc_lr:.4f}")
print(f"Precision: {pre_lr:.4f}")
print(f"Recall   : {rec_lr:.4f}")
print(f"F1-Score : {f1_lr:.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_te, pred_lr))
print("\nClassification Report:\n", classification_report(y_te, pred_lr, target_names=enc.classes_))


Training Logistic Regression...

-------------------------------------------------------
LOGISTIC REGRESSION (TF-IDF) — RESULTS
-------------------------------------------------------
Accuracy : 0.8650
Precision: 0.8650
Recall   : 0.8650
F1-Score : 0.8650

Confusion Matrix:
 [[439  68]
 [ 67 426]]

Classification Report:
               precision    recall  f1-score   support

    negative       0.87      0.87      0.87       507
    positive       0.86      0.86      0.86       493

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.87      0.86      0.87      1000



In [17]:
# =========================
# CELL 11 — Build + Train LSTM (Optimized)
# =========================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight

# class weights (balanced)
cw = compute_class_weight(class_weight="balanced", classes=np.unique(y_tr), y=y_tr)
cw_map = {i: cw[i] for i in range(len(cw))}

print("🧠 Building LSTM network...")
net = Sequential([
    Embedding(input_dim=VOCAB_LIMIT, output_dim=100, input_length=SEQ_LEN),
    SpatialDropout1D(0.30),

    Bidirectional(LSTM(
        128,
        dropout=0.50,
        recurrent_dropout=0.40,
        return_sequences=False
    )),

    Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.50),

    Dense(32, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.30),

    Dense(len(enc.classes_), activation="softmax")
])

net.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

net.summary()

early = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
    min_delta=0.001,
    verbose=1
)

reduce = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)

print("\n🚀 Training LSTM...")
hist = net.fit(
    Xtr_seq, y_tr,
    epochs=20,
    batch_size=32,
    validation_split=0.15,
    class_weight=cw_map,
    callbacks=[early, reduce],
    verbose=1
)

print("✅ LSTM training done.")


🧠 Building LSTM network...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


🚀 Training LSTM...
Epoch 1/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 46s 359ms/step - accuracy: 0.4966 - loss: 0.7889 - val_accuracy: 0.5333 - val_loss: 0.7457 - learning_rate: 0.0010
Epoch 2/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 58s 542ms/step - accuracy: 0.5696 - loss: 0.7139 - val_accuracy: 0.7433 - val_loss: 0.5938 - learning_rate: 0.0010
Epoch 3/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 58s 540ms/step - accuracy: 0.7878 - loss: 0.5101 - val_accuracy: 0.7417 - val_loss: 0.5476 - learning_rate: 0.0010
Epoch 4/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 57s 531ms/step - accuracy: 0.8619 - loss: 0.3836 - val_accuracy: 0.7733 - val_loss: 0.5524 - learning_rate: 0.0010
Epoch 5/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 53s 498ms/step - accuracy: 0.8987 - loss: 0.3079 - val_accuracy: 0.7683 - val_loss: 0.5815 - learning_rate: 0.0010
Epoch 6/20
107/107 ━━━━━━━━━━━━━━━━━━━━ 0s 516ms/step - accuracy: 0.9258 - loss: 0.2553
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
107/107 ━━━━━━━━━━━━━━━━━━━━ 56s 526ms/st

In [18]:
# Evaluate LSTM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

probs = net.predict(Xte_seq, verbose=0)
pred_lstm = np.argmax(probs, axis=1)

acc_lstm = accuracy_score(y_te, pred_lstm)
pre_lstm = precision_score(y_te, pred_lstm, average="weighted", zero_division=0)
rec_lstm = recall_score(y_te, pred_lstm, average="weighted", zero_division=0)
f1_lstm  = f1_score(y_te, pred_lstm, average="weighted", zero_division=0)

print("\n" + "-"*55)
print("LSTM (Optimized) — RESULTS")
print("-"*55)
print(f"Accuracy : {acc_lstm:.4f}")
print(f"Precision: {pre_lstm:.4f}")
print(f"Recall   : {rec_lstm:.4f}")
print(f"F1-Score : {f1_lstm:.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_te, pred_lstm))
print("\nClassification Report:\n", classification_report(y_te, pred_lstm, target_names=enc.classes_))


-------------------------------------------------------
LSTM (Optimized) — RESULTS
-------------------------------------------------------
Accuracy : 0.7660
Precision: 0.7665
Recall   : 0.7660
F1-Score : 0.7658

Confusion Matrix:
 [[402 105]
 [129 364]]

Classification Report:
               precision    recall  f1-score   support

    negative       0.76      0.79      0.77       507
    positive       0.78      0.74      0.76       493

    accuracy                           0.77      1000
   macro avg       0.77      0.77      0.77      1000
weighted avg       0.77      0.77      0.77      1000



In [19]:
import os
import pickle

print("\nSaving models & preprocessors...")
os.makedirs("saved_models", exist_ok=True)

# 1) Save Logistic Regression + TF-IDF
with open("saved_models/lr_model.pkl", "wb") as f:
    pickle.dump(lr_clf, f)

with open("saved_models/vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vec, f)

# 2) Save LSTM (both formats)
net.save("saved_models/lstm_model.keras")
net.save("saved_models/lstm_model.h5")

# 3) Save preprocessing objects
with open("saved_models/tokenizer.pkl", "wb") as f:
    pickle.dump(tok, f)

with open("saved_models/label_encoder.pkl", "wb") as f:
    pickle.dump(enc, f)

with open("saved_models/lemmatizer.pkl", "wb") as f:
    pickle.dump(LEM, f)

with open("saved_models/stopwords.pkl", "wb") as f:
    pickle.dump(STOP_SET, f)

# 4) Save parameters
with open("saved_models/lstm_params.pkl", "wb") as f:
    pickle.dump({"max_words": VOCAB_LIMIT, "max_len": SEQ_LEN}, f)

with open("saved_models/all_params.pkl", "wb") as f:
    pickle.dump({"max_words": VOCAB_LIMIT, "max_len": SEQ_LEN, "stop_words": STOP_SET}, f)

# 5) Save metrics
metrics_pack = {
    "logistic_regression": {"accuracy": acc_lr, "precision": pre_lr, "recall": rec_lr, "f1_score": f1_lr},
    "lstm": {"accuracy": acc_lstm, "precision": pre_lstm, "recall": rec_lstm, "f1_score": f1_lstm},
}
with open("saved_models/metrics.pkl", "wb") as f:
    pickle.dump(metrics_pack, f)

print("Done! All files saved in: saved_models/")



Saving models & preprocessors...


Done! All files saved in: saved_models/
